
# Step 5 — Dataset Split & Fine‑Tuning Preparation

This step prepares the dataset for fine‑tuning.

Pipeline:

1. Load cleaned dataset from **Step‑3**
2. Generate prompts
3. Split dataset into **training (80%)** and **test (20%)**
4. Create **train.jsonl** for fine‑tuning
5. Save **test_set.json** for evaluation in Step‑7

Using a train/test split prevents **data leakage** and ensures the model is evaluated on data it has never seen before.


In [ ]:

import json
import sys
import os
import random

sys.path.append(os.path.abspath(".."))

from src.job_item import JobItem


## Load cleaned dataset

In [ ]:

with open("../data/processed/jobs_cleaned.json") as f:
    data = json.load(f)

items = [JobItem(**row) for row in data]

print("Dataset size:", len(items))


## Generate prompts for each job

In [ ]:

for item in items:
    item.make_prompt()

print(items[0].prompt)


## Split dataset (80% train / 20% test)

In [ ]:

random.shuffle(items)

split_index = int(len(items) * 0.8)

train_items = items[:split_index]
test_items = items[split_index:]

print("Train size:", len(train_items))
print("Test size:", len(test_items))


## Create fine‑tuning dataset

In [ ]:

training_data = []

for item in train_items:

    example = {
        "messages": [
            {"role": "user", "content": item.test_prompt()},
            {"role": "assistant", "content": str(round(item.salary))}
        ]
    }

    training_data.append(example)

print("Training examples:", len(training_data))


## Save train.jsonl (used for model training)

In [ ]:

os.makedirs("../data/fine_tune", exist_ok=True)

with open("../data/fine_tune/train.jsonl", "w") as f:
    for row in training_data:
        f.write(json.dumps(row) + "\n")

print("Saved: data/fine_tune/train.jsonl")


## Save test dataset (used for evaluation later)

In [ ]:

with open("../data/fine_tune/test_set.json", "w") as f:
    json.dump([i.model_dump() for i in test_items], f, indent=2)

print("Saved: data/fine_tune/test_set.json")


## Inspect sample training rows

In [ ]:

with open("../data/fine_tune/train.jsonl") as f:
    for _ in range(3):
        print(f.readline())
